In [18]:
import os
import glob
import pandas as pd
import pickle
from konlpy.tag import Mecab
from collections import Counter
from tqdm import tqdm

# ---------------------------------------------------------
# 1. 설정 및 Mecab 초기화
# ---------------------------------------------------------
mecab = Mecab(dicpath='/opt/homebrew/lib/mecab/dic/mecab-ko-dic') 

TARGET_DIRS = [
    "data/대법원_판례_txt화", 
    "data/법률_정제_완료_txt화"
]

OUTPUT_FILE_PKL = "legal_corpus_stats_all_pos.pkl" 
OUTPUT_FILE_CSV = "legal_corpus_stats_all_pos.csv"

# ---------------------------------------------------------
# 2. 전처리 함수 (수정됨: 모든 형태소 반환)
# ---------------------------------------------------------
def preprocess_text(text):
    """
    Mecab을 사용하여 모든 형태소와 품사 태그를 반환
    Returns: list of tuples [('단어', '품사'), ...]
    """
    if not isinstance(text, str): return []
    
    try:
        # mecab.pos()는 [('나', 'NP'), ('는', 'JX'), ...] 형태를 반환함
        # 품사 필터링 없이 그대로 반환
        return mecab.pos(text)
    except Exception as e:
        print(f"Mecab Error: {e}")
        return []

# ---------------------------------------------------------
# 3. 데이터 순회 및 카운팅 (수정됨: (단어, 품사) 쌍 카운팅)
# ---------------------------------------------------------
def process_corpus(target_dirs):
    # 키(Key)가 이제 'string'이 아니라 ('word', 'pos') 튜플이 됨
    total_tf = Counter() 
    total_df = Counter() 
    doc_count = 0 
    
    all_files = []
    for d in target_dirs:
        # txt 파일 기준 (필요시 json 등으로 변경)
        files = glob.glob(os.path.join(d, "**", "*.txt"), recursive=True)
        all_files.extend(files)
        
    print(f"총 {len(all_files)}개의 파일을 발견했습니다. 전체 형태소 분석을 시작합니다...")
    
    for file_path in tqdm(all_files):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='cp949') as f:
                    content = f.read()
            except:
                continue 

        tokens = preprocess_text(content) # [(word, pos), (word, pos)...]
        
        if not tokens: continue

        # 1. TF (단어, 품사) 쌍 카운트
        total_tf.update(tokens)
        
        # 2. DF (문서 내 중복 제거 후 카운트)
        unique_tokens = set(tokens)
        total_df.update(unique_tokens)
        
        doc_count += 1

    return total_tf, total_df, doc_count

# ---------------------------------------------------------
# 4. 실행 및 저장 (수정됨: POS 컬럼 추가)
# ---------------------------------------------------------
if __name__ == "__main__":
    tf_counter, df_counter, total_docs = process_corpus(TARGET_DIRS)
    
    print("-" * 50)
    print(f"분석 완료! (총 문서 수: {total_docs})")
    print(f"발견된 고유 형태소(단어+품사) 수: {len(tf_counter)}")
    
    # 데이터프레임 변환
    data = []
    # tf_counter의 키는 ('단어', '품사') 튜플임
    for (word, pos), tf in tf_counter.items():
        df = df_counter[(word, pos)]
        data.append({
            'word': word, 
            'pos': pos,       # 품사 컬럼 추가
            'tf': tf, 
            'df': df
        })
    
    result_df = pd.DataFrame(data)
    result_df['total_docs'] = total_docs
    
    # 저장
    print("파일 저장 중...")
    result_df.to_pickle(OUTPUT_FILE_PKL)
    result_df.to_csv(OUTPUT_FILE_CSV, index=False, encoding='utf-8-sig')
    
    print(f"저장 완료:")
    print(f" - Pickle: {OUTPUT_FILE_PKL}")
    print(f" - CSV:    {OUTPUT_FILE_CSV}")
    
    # 상위 빈도 예시 출력
    print("\n[Top 10 빈도 형태소]")
    print(result_df.sort_values(by='tf', ascending=False).head(10)[['word', 'pos', 'tf', 'df']])

총 7162개의 파일을 발견했습니다. 전체 형태소 분석을 시작합니다...


100%|██████████| 7162/7162 [00:41<00:00, 171.20it/s]


--------------------------------------------------
분석 완료! (총 문서 수: 7162)
발견된 고유 형태소(단어+품사) 수: 99550
파일 저장 중...
저장 완료:
 - Pickle: legal_corpus_stats_all_pos.pkl
 - CSV:    legal_corpus_stats_all_pos.csv

[Top 10 빈도 형태소]
   word  pos      tf    df
76    에  JKB  908688  7159
26    의  JKG  903720  7159
43    .   SF  645055  7162
61    을  JKO  550712  6894
63    하  XSV  546234  7155
89    제  XPN  486704  6786
51    를  JKO  378164  7153
2     ,   SC  370237  7143
83    는  ETM  358647  6874
40    .   SY  341159  6909
